In [1]:
import numpy as np

# Define states and observations
states = ['S1=/f/', 'S2=/oʊ/', 'S3=/r/', 'S4=End']
num_states = len(states)

# Observation sequence
O = ['O1', 'O2', 'O3', 'O4']
T = len(O)  # Number of time steps

# Initialize delta and psi matrices
delta = np.zeros((num_states, T))  # Best path probabilities
psi = np.zeros((num_states, T), dtype=int)  # Backpointers for path reconstruction

# Answer for part (a)
print("=== Part (a): Initialization at t=1 ===")
print("General formula: δ₁(i) = π(i) * b_i(O₁)")
print("Where:")
print("  - π(i) is initial probability of state i")
print("  - b_i(O₁) is probability of observation O₁ given state i")
print("\nIn strictly ordered pronunciation model:")
print("δ₁(i) = 0 for all states except S1 because:")
print("1. π(S1) = 1 (we always start with /f/)")
print("2. π(i) = 0 for i ≠ S1 (cannot start with other phonemes)")
print("3. So δ₁(i) = 0 for i ≠ S1 regardless of b_i(O₁)")

=== Part (a): Initialization at t=1 ===
General formula: δ₁(i) = π(i) * b_i(O₁)
Where:
  - π(i) is initial probability of state i
  - b_i(O₁) is probability of observation O₁ given state i

In strictly ordered pronunciation model:
δ₁(i) = 0 for all states except S1 because:
1. π(S1) = 1 (we always start with /f/)
2. π(i) = 0 for i ≠ S1 (cannot start with other phonemes)
3. So δ₁(i) = 0 for i ≠ S1 regardless of b_i(O₁)


In [2]:
# Transition probabilities (a_ij = P(state j at t+1 | state i at t))
# Format: a[i][j] = probability from state i to state j
A = np.array([
    [0.0, 0.8, 0.0, 0.2],   # From S1: can go to S2 or End (with small prob)
    [0.0, 0.3, 0.6, 0.1],   # From S2: can go to S3 or stay or End
    [0.0, 0.0, 0.4, 0.6],   # From S3: can go to End or stay
    [0.0, 0.0, 0.0, 1.0]    # From End: absorbing state
])

# Observation probabilities (b_i(o) = P(observation o | state i))
# Format: b[state_index][observation_index]
B = np.array([
    [0.7, 0.2, 0.1, 0.0],   # S1 probabilities for O1, O2, O3, O4
    [0.1, 0.6, 0.2, 0.1],   # S2 probabilities
    [0.0, 0.2, 0.7, 0.1],   # S3 probabilities
    [0.0, 0.0, 0.0, 1.0]    # End probabilities
])

# Initial state probabilities
pi = np.array([1.0, 0.0, 0.0, 0.0])  # Only start with S1

print("Model Parameters:")
print(f"Transition matrix A:\n{A}")
print(f"\nEmission matrix B:\n{B}")
print(f"\nInitial probabilities π: {pi}")

Model Parameters:
Transition matrix A:
[[0.  0.8 0.  0.2]
 [0.  0.3 0.6 0.1]
 [0.  0.  0.4 0.6]
 [0.  0.  0.  1. ]]

Emission matrix B:
[[0.7 0.2 0.1 0. ]
 [0.1 0.6 0.2 0.1]
 [0.  0.2 0.7 0.1]
 [0.  0.  0.  1. ]]

Initial probabilities π: [1. 0. 0. 0.]


In [3]:
# Time t = 1 initialization
print("=== Initialization (t=1) ===")
for i in range(num_states):
    delta[i, 0] = pi[i] * B[i, 0]  # b_i(O1) - using O1 as index 0
    psi[i, 0] = 0  # No previous state at t=1
    print(f"δ₁({states[i]}) = π({states[i]}) × b_{i}(O₁) = {pi[i]:.2f} × {B[i,0]:.2f} = {delta[i,0]:.2f}")

print(f"\nResult: δ₁ = {delta[:, 0]}")
print("Only δ₁(S1) is non-zero, confirming strictly ordered pronunciation model.")

=== Initialization (t=1) ===
δ₁(S1=/f/) = π(S1=/f/) × b_0(O₁) = 1.00 × 0.70 = 0.70
δ₁(S2=/oʊ/) = π(S2=/oʊ/) × b_1(O₁) = 0.00 × 0.10 = 0.00
δ₁(S3=/r/) = π(S3=/r/) × b_2(O₁) = 0.00 × 0.00 = 0.00
δ₁(S4=End) = π(S4=End) × b_3(O₁) = 0.00 × 0.00 = 0.00

Result: δ₁ = [0.7 0.  0.  0. ]
Only δ₁(S1) is non-zero, confirming strictly ordered pronunciation model.


In [4]:
# Induction step for t = 2 to T
print("\n=== Part (b): Induction Step ===")
print("Key difference from Forward Algorithm:")
print("1. Forward algorithm uses SUM over all paths: α_t(j) = Σ_i [α_{t-1}(i) × a_ij] × b_j(O_t)")
print("2. Viterbi uses MAX over paths: δ_t(j) = max_i [δ_{t-1}(i) × a_ij] × b_j(O_t)")
print("\nWhy maximization?")
print("To find SINGLE most probable path Q* (Viterbi path), not sum of ALL paths")
print("Maximization tracks the BEST path to each state, discarding inferior paths")

# Perform induction
for t in range(1, T):
    print(f"\n--- Time t={t+1}, Observation O{t+1} ---")
    for j in range(num_states):
        # Find the maximum over all possible previous states i
        max_val = -1
        max_idx = -1
        for i in range(num_states):
            val = delta[i, t-1] * A[i, j]
            if val > max_val:
                max_val = val
                max_idx = i
        
        delta[j, t] = max_val * B[j, t]  # b_j(O_t)
        psi[j, t] = max_idx
        
        print(f"δ_{t+1}({states[j]}) = max_i[δ_{t}(i)×a_ij] × b_j(O{t+1})")
        print(f"  = {max_val:.3f} × {B[j,t]:.2f} = {delta[j,t]:.3f}")
        print(f"  Best previous state: {states[max_idx]} (ψ_{t+1}({states[j]}) = {max_idx})")

print(f"\nDelta matrix after induction:\n{delta}")
print(f"\nPsi matrix (backpointers):\n{psi}")


=== Part (b): Induction Step ===
Key difference from Forward Algorithm:
1. Forward algorithm uses SUM over all paths: α_t(j) = Σ_i [α_{t-1}(i) × a_ij] × b_j(O_t)
2. Viterbi uses MAX over paths: δ_t(j) = max_i [δ_{t-1}(i) × a_ij] × b_j(O_t)

Why maximization?
To find SINGLE most probable path Q* (Viterbi path), not sum of ALL paths
Maximization tracks the BEST path to each state, discarding inferior paths

--- Time t=2, Observation O2 ---
δ_2(S1=/f/) = max_i[δ_1(i)×a_ij] × b_j(O2)
  = 0.000 × 0.20 = 0.000
  Best previous state: S1=/f/ (ψ_2(S1=/f/) = 0)
δ_2(S2=/oʊ/) = max_i[δ_1(i)×a_ij] × b_j(O2)
  = 0.560 × 0.60 = 0.336
  Best previous state: S1=/f/ (ψ_2(S2=/oʊ/) = 0)
δ_2(S3=/r/) = max_i[δ_1(i)×a_ij] × b_j(O2)
  = 0.000 × 0.20 = 0.000
  Best previous state: S1=/f/ (ψ_2(S3=/r/) = 0)
δ_2(S4=End) = max_i[δ_1(i)×a_ij] × b_j(O2)
  = 0.140 × 0.00 = 0.000
  Best previous state: S1=/f/ (ψ_2(S4=End) = 0)

--- Time t=3, Observation O3 ---
δ_3(S1=/f/) = max_i[δ_2(i)×a_ij] × b_j(O3)
  = 0.000 × 0.